# CDAC — Sizing Array Kapasitor
### SAR ADC 8-bit 10 MS/s differential, sky130A, Tiny Tapeout 1x2

Skema switching: **split-monotonic** langkah k=1..N_SPLIT, **monotonic biasa** sisanya.

**Referensi skema:** Liu et al., *"A 10-bit 50-MS/s SAR ADC With a Monotonic Capacitor
Switching Procedure,"* JSSC 2010 — untuk monotonic. Untuk split: keluarga
capacitor-splitting dengan V_CM hampir konstan (JCSC 2020; ISCAS 2018).

---

## Kenapa notebook ini terpisah dari `comparator.ipynb`

Arah ketergantungannya **satu arah** dan tidak boleh dibalik:

```
spec  ->  CDAC (C_u dari spec DNL)  ->  sisa anggaran error  ->  comparator
```

`C_u` **tidak bergantung sama sekali** pada anggaran noise — ia keluar dari spec DNL.
Sebaliknya, jatah noise comparator adalah **sisa** setelah CDAC mengambil bagiannya.
Jadi notebook ini bisa jalan sendiri; `comparator.ipynb` tidak bisa.

Antarmukanya satu file: notebook ini menulis `cdac_result.json`, dan `comparator.ipynb`
membacanya lewat `adc_spec.load_cdac()`. Kalau file itu belum ada, `comparator.ipynb`
**menolak jalan** — bukan memakai angka karangan.

## ATURAN NOTEBOOK INI

| | Kategori | Cara membaca |
|---|---|---|
| **(a)** | **Spec keras** | `adc_spec.py`. Diberikan, tidak dinegosiasikan |
| **(b)** | **Fakta PDK / hasil MC** | Dibaca dari file model, DRC deck, atau dihitung MC di sini |
| **(c)** | **Pilihan desain** | `N_SPLIT`, `CAPM_SIDE`. Konsekuensinya di-sweep |
| **(d)** | **BELUM DIKETAHUI** | `None`. Notebook menolak menghitung yang bergantung padanya |

Satu koefisien yang dulu diturunkan tangan dan **terbukti salah** ada di Langkah 4.
Ia sekarang dihitung MC di dalam notebook ini, bukan diketik.

In [1]:
import numpy as np
import matplotlib.pyplot as plt

from adc_spec import *          # (a) spec, (b) fakta PDK kapasitor, (c) T_SAMPLE & N_SPLIT
from adc_spec import c_unit, sigma_unit, save_cdac, summary

np.set_printoptions(precision=4, suppress=True)
print(summary())

# ===== (d) BELUM DIKETAHUI ===================================================
DVOS_DVCM = None    # V/V. Sensitivitas offset comparator terhadap V_CM.
                    # Menentukan apakah N_SPLIT=4 cukup. Sweep DC di comparator.sch.
RIPPLE_VREF = None  # V rms. Kopling ke differensial dihitung di Langkah 8.
SIGMA_JITTER = None # s rms. Aperture jitter clock TT.
C_PAR_TOP = 0.0     # F. Parasitik top plate di luar gerbang comparator.
                    # 0 = batas bawah PRA-LAYOUT. Ganti dgn PEX.
print(f'\n(d) kosong: DVOS_DVCM={DVOS_DVCM}  RIPPLE_VREF={RIPPLE_VREF}  '
      f'SIGMA_JITTER={SIGMA_JITTER}')
print(f'    C_PAR_TOP={C_PAR_TOP*1e15:.1f} fF  <- batas bawah pra-layout, bukan nilai sebenarnya')

SPEC  N=8  fs=10 MS/s  VREF=1.8 V  ENOB=7.5
      LSB      = 14.0625 mV differential
      V_CM     = 0.900000 V (dipaksa) .. 0.850781 V (droop 49.2188 mV = 3.500 LSB)
      t_trial  = 10.625 ns
      V_circ   = 4.0646 mV  (V_q = 4.0595 mV di luar ini)
      7 langkah DAC, bobot [64, 32, 16, 8, 4, 2, 1] + dummy 1 C_u = 128 C_u/sisi

(d) kosong: DVOS_DVCM=None  RIPPLE_VREF=None  SIGMA_JITTER=None
    C_PAR_TOP=0.0 fF  <- batas bawah pra-layout, bukan nilai sebenarnya


## Langkah 1 — Struktur array: jumlah langkah dan bobot

Top-plate sampling membuat comparison ke-1 **gratis**, jadi langkah DAC ada `N-1 = 7`,
bukan 8. Langkah `k` dijalankan **sesudah** comparison ke-`k`.

Pencarian biner pada residu +/- VREF: threshold comparison ke-(k+1) bergeser `VREF/2^k`.

```
step_diff(k) = VREF/2^k
w(k)         = step_diff(k)/LSB = (VREF/2^k)/(2*VREF/2^N) = 2^(N-1)/2^k
C_tot,sisi   = 2^(N-1)*C_u
```

`C_tot` **wajib** 2^(N-1) persis, supaya `w/C_tot` selalu pecahan biner eksak. Sisa
`2^(N-1) - sum(w)` diisi kapasitor **dummy**, dan dummy itu bukan hiasan.

In [2]:
print(f'n_step = N - 1 = {n_step}   (comparison ke-1 gratis)')
print(f'C_tot,sisi = 2^(N-1) = {n_unit} C_u\n')
print(' k  mode    step_diff(mV)   w(C_u)   VREF*w/C_tot (mV)   cek')
for k, w in enumerate(w_step, start=1):
    sd = VREF/2**k
    chk = VREF*w/n_unit
    mode = 'split' if k <= N_SPLIT else 'mono'
    print(f' {k}  {mode:6s}  {sd*1e3:11.4f}   {w:6d}   {chk*1e3:14.4f}   '
          f'{"OK" if abs(chk-sd) < 1e-12 else "SALAH"}')
print(f'\nsum w = {sum(w_step)} C_u  ->  C_dummy = {n_unit} - {sum(w_step)} = {w_dummy} C_u')
print(f'  step k=1 tanpa dummy: VREF*{w_step[0]}/{sum(w_step)} = '
      f'{VREF*w_step[0]/sum(w_step)*1e3:.4f} mV  <- bukan 900 mV. Bobot biner rusak.')
print(f'  step terkecil = {VREF*w_step[-1]/n_unit*1e3:.4f} mV = '
      f'{VREF*w_step[-1]/n_unit/LSB:.1f} LSB')

n_step = N - 1 = 7   (comparison ke-1 gratis)
C_tot,sisi = 2^(N-1) = 128 C_u

 k  mode    step_diff(mV)   w(C_u)   VREF*w/C_tot (mV)   cek
 1  split      900.0000       64         900.0000   OK
 2  split      450.0000       32         450.0000   OK
 3  split      225.0000       16         225.0000   OK
 4  split      112.5000        8         112.5000   OK
 5  mono        56.2500        4          56.2500   OK
 6  mono        28.1250        2          28.1250   OK
 7  mono        14.0625        1          14.0625   OK

sum w = 127 C_u  ->  C_dummy = 128 - 127 = 1 C_u
  step k=1 tanpa dummy: VREF*64/127 = 907.0866 mV  <- bukan 900 mV. Bobot biner rusak.
  step terkecil = 14.0625 mV = 1.0 LSB


## Langkah 2 — Identitas split, dan droop V_CM yang tersisa

Konservasi muatan pada top plate yang mengapung memberi satu persamaan untuk semuanya:

```
dV_top = -(C_geser/C_tot) * dV_bottom
```

Split langkah `k` memecah bobot `w` jadi dua paruh `w/2`, satu ditarik ke bawah di satu
sisi, satu didorong ke atas di sisi lain:

```
dV_p = -(w/2)/C_tot*VREF        dV_n = +(w/2)/C_tot*VREF
  differential = dV_p - dV_n     = -w/C_tot*VREF     <- IDENTIK dgn monotonic bobot w
  common mode  = (dV_p + dV_n)/2 = 0                 <- seluruh trik ada di sini
```

Step differential utuh karena differential menjumlahkan dua kontribusi yang saling
menguatkan, sementara CM merata-ratakan dua kontribusi yang saling membatalkan.

Batas geometris: paruh harus bilangan bulat unit. Hanya langkah dengan `w = 1` yang tidak
bisa dipecah, jadi `N_SPLIT` maksimum = `n_step - 1`.

In [3]:
print('pemecahan paruh, dan batas geometrisnya:')
for k, w in enumerate(w_step, start=1):
    ok = (w % 2 == 0)
    tag = f'{w//2} + {w//2}' if ok else f'{w/2} + {w/2}  <- TIDAK BISA (bukan bilangan bulat)'
    print(f'  k={k}: {w:3d} C_u -> {tag}')
print(f'\n  -> N_SPLIT maksimum secara geometri = {n_step-1}; dipakai N_SPLIT = {N_SPLIT} (pilihan)')

print('\ndroop V_CM vs N_SPLIT:')
full = sum(VREF/2**(k+1) for k in range(1, n_step+1))
for ns in range(0, n_step):
    d = sum(VREF/2**(k+1) for k in range(ns+1, n_step))
    star = '  <== dipakai' if ns == N_SPLIT else ''
    print(f'  N_SPLIT={ns}: droop {d*1e3:9.4f} mV = {d/LSB:6.3f} LSB  '
          f'V_CM 0.900000 -> {V_CM_MAX-d:.6f} V  reduksi {100*(1-d/full):5.2f}%{star}')

print(f'\nlangkah monotonic yang menyumbang droop (N_SPLIT={N_SPLIT}):')
for k, s, d in cm_steps:
    print(f'  langkah {k}: step 1 sisi {s*1e3:8.4f} mV -> CM turun {d*1e3:8.4f} mV')
print(f'  total {dV_CM*1e3:.4f} mV = {dV_CM/LSB:.3f} LSB -> V_CM_MIN = {V_CM_MIN:.6f} V')
print('\n  Catatan: split yang paruhnya TIDAK sama besar lebih buruk daripada tidak split,')
print('  karena kamu bayar dua driver dan tetap dapat droop. Cek ini di skematik.')

pemecahan paruh, dan batas geometrisnya:
  k=1:  64 C_u -> 32 + 32
  k=2:  32 C_u -> 16 + 16
  k=3:  16 C_u -> 8 + 8
  k=4:   8 C_u -> 4 + 4
  k=5:   4 C_u -> 2 + 2
  k=6:   2 C_u -> 1 + 1
  k=7:   1 C_u -> 0.5 + 0.5  <- TIDAK BISA (bukan bilangan bulat)

  -> N_SPLIT maksimum secara geometri = 6; dipakai N_SPLIT = 4 (pilihan)

droop V_CM vs N_SPLIT:
  N_SPLIT=0: droop  885.9375 mV = 63.000 LSB  V_CM 0.900000 -> 0.014062 V  reduksi  0.79%
  N_SPLIT=1: droop  435.9375 mV = 31.000 LSB  V_CM 0.900000 -> 0.464062 V  reduksi 51.18%
  N_SPLIT=2: droop  210.9375 mV = 15.000 LSB  V_CM 0.900000 -> 0.689063 V  reduksi 76.38%
  N_SPLIT=3: droop   98.4375 mV =  7.000 LSB  V_CM 0.900000 -> 0.801563 V  reduksi 88.98%
  N_SPLIT=4: droop   42.1875 mV =  3.000 LSB  V_CM 0.900000 -> 0.857812 V  reduksi 95.28%  <== dipakai
  N_SPLIT=5: droop   14.0625 mV =  1.000 LSB  V_CM 0.900000 -> 0.885938 V  reduksi 98.43%
  N_SPLIT=6: droop    0.0000 mV =  0.000 LSB  V_CM 0.900000 -> 0.900000 V  reduksi 100.00%

la

## Langkah 3 — Constraint kT/C: dihitung, lalu dibuang

Dua sisi disampling independen, jadi varians dijumlah:

```
sigma_kTC^2 = 2*k*T / C_tot,sisi
```

Batas paling longgar yang bisa dibayangkan: berikan **seluruh** `V_circ` ke suku ini.

In [4]:
C_tot_ktc_min = 2*kT/Vcirc**2
print(f'Kalau SELURUH V_circ = {Vcirc*1e3:.4f} mV diberikan ke kT/C:')
print(f'  C_tot,sisi >= 2*k*T/V_circ^2 = {2*kT:.6e}/{Vcirc**2:.6e} = {C_tot_ktc_min*1e15:.4f} fF')
print(f'  C_u        >= {C_tot_ktc_min*1e15:.4f}/{n_unit} = {C_tot_ktc_min/n_unit*1e18:.2f} aF')
print(f'\n  Kapasitor DRC terkecil (capm {CAPM_W_MIN}x{CAPM_W_MIN} um) = '
      f'{c_unit(CAPM_W_MIN)*1e15:.4f} fF')
print(f'  -> lebih besar {c_unit(CAPM_W_MIN)/(C_tot_ktc_min/n_unit):.0f}x dari yang dibutuhkan.')
print('  kT/C BUKAN constraint di ADC ini. Ia tidak akan muncul lagi sebagai penggerak.')
print(f'\n  Sebabnya C ~ 1/LSB^2 = 2^(2N)/(4*VREF^2). Naik ke 12 bit pada VREF sama:')
print(f'    LSB   = {2*VREF/2**12*1e3:.4f} mV  (dari {LSB*1e3:.4f} mV)')
print(f'    C butuh naik {2**(2*(12-N)):.0f}x -> orde pF, dan di sana kT/C mulai menggigit.')

Kalau SELURUH V_circ = 4.0646 mV diberikan ke kT/C:
  C_tot,sisi >= 2*k*T/V_circ^2 = 8.283894e-21/1.652059e-05 = 0.5014 fF
  C_u        >= 0.5014/128 = 3.92 aF

  Kapasitor DRC terkecil (capm 1.0x1.0 um) = 2.6422 fF
  -> lebih besar 674x dari yang dibutuhkan.
  kT/C BUKAN constraint di ADC ini. Ia tidak akan muncul lagi sebagai penggerak.

  Sebabnya C ~ 1/LSB^2 = 2^(2N)/(4*VREF^2). Naik ke 12 bit pada VREF sama:
    LSB   = 0.8789 mV  (dari 14.0625 mV)
    C butuh naik 256x -> orde pF, dan di sana kT/C mulai menggigit.


## Langkah 4 — Matching: **turunan tangan diverifikasi Monte Carlo**

```
sigma_r = sigma_Cu/C_u = A_C/sqrt(A_u)          A_C = 2.8 %*um  (fakta PDK)
```

### Turunan DNL mid-code (satu-satunya yang punya bentuk tertutup)

Threshold comparison ke-1 ada di 0 V dan dibangun dari **nol** kapasitor — errornya
tepat nol. Itu sifat khas top-plate sampling. Threshold tetangganya di +1 LSB dibangun
dari `+s1 -s2 -s3 -s4 -s5 -s6 -s7`, yaitu **seluruh** unit:

```
sigma_DNL,mid = sqrt(sum w(k)) * sigma_r = sqrt(127)*sigma_r = 11.2694*sigma_r   [LSB]
```

Jadi DNL terburuk ada di **mid-code**, tempat error seluruh array dilawan threshold yang
errornya nol.

### Yang TIDAK punya bentuk tertutup

`INL_rms`, `E[max|DNL|]`, `E[max|INL|]` tidak. Turunan tangan saya yang lama:

```
var(INL(u)) = 127*(1-u^2)*sigma_r^2   ->   INL_rms = sqrt(127*2/3)*sigma_r = 9.2014*sigma_r
```

**Itu salah.** Ia mengasumsikan ke-255 threshold memakai 127 unit. Yang benar, threshold
di level `m` memakai `2^(N-1)*(1-2^-(m-1))` unit — dan threshold level 1 memakai **nol**.
Sel di bawah menghitung profilnya, lalu MC memberi koefisien yang benar.

Metodenya: 255 node pohon keputusan dihitung sebagai fungsi **linear** nilai kapasitor,
jadi seluruh MC jadi dua perkalian matriks. `threshold = dV_n - dV_p` pada residu nol.

In [5]:
print('profil varians per level threshold -- kenapa rumus tangan itu salah:')
print('  level m   n_threshold   n_unit terlibat = 2^(N-1)*(1-2^-(m-1))')
for m in range(1, N+1):
    print(f'     {m}          {2**(m-1):4d}          {int(n_unit*(1-2.0**-(m-1))):4d}')
print(f'  level 1 memakai NOL unit; hanya level {N} yang memakai {sum(w_step)}.')
print(f'  rata-rata berbobot = '
      f'{sum(2**(m-1)*int(n_unit*(1-2.0**-(m-1))) for m in range(1,N+1))/(2**N-1):.2f} unit')


def build_maps(n=N, ns=N_SPLIT, vref=VREF):
    '''Matriks dB (2^N-1 x n_unit) per sisi, nilai dalam {-VREF, 0, +VREF}
    relatif preset. Baris = satu node pohon keputusan.'''
    nu = 2**(n-1)
    w = [nu//2**k for k in range(1, n)]
    grp, o = {}, 0
    for k in range(1, n):
        if k <= ns:
            grp[('A', k)] = (o, o+w[k-1]//2); o += w[k-1]//2
            grp[('B', k)] = (o, o+w[k-1]//2); o += w[k-1]//2
        else:
            grp[('M', k)] = (o, o+w[k-1]);    o += w[k-1]
    grp[('D', 0)] = (o, o+1); o += 1
    assert o == nu, f'peta grup tidak menutup {nu} unit (dapat {o})'
    paths = [[(v >> (m-2-i)) & 1 for i in range(m-1)]
             for m in range(1, n+1) for v in range(2**(m-1))]
    Mp = np.zeros((len(paths), nu)); Mn = np.zeros_like(Mp)
    for i, b in enumerate(paths):
        for k, bk in enumerate(b, start=1):
            if k <= ns:
                a, bb = grp[('A', k)], grp[('B', k)]
                if bk: Mp[i, a[0]:a[1]] = -vref;  Mn[i, bb[0]:bb[1]] = +vref
                else:  Mp[i, bb[0]:bb[1]] = +vref; Mn[i, a[0]:a[1]] = -vref
            else:
                m_ = grp[('M', k)]
                if bk: Mp[i, m_[0]:m_[1]] = -vref
                else:  Mn[i, m_[0]:m_[1]] = -vref
    return Mp, Mn, grp


def mc_linearity(sigma_r, M=20000, seed=0, batch=2000):
    Mp, Mn, _ = build_maps()
    T0 = (np.ones(n_unit) @ Mn.T - np.ones(n_unit) @ Mp.T)/n_unit
    i0 = int(np.where(np.argsort(T0) == int(np.argmin(np.abs(T0))))[0][0])
    rng = np.random.default_rng(seed); acc = []
    for s in range(0, M, batch):
        b = min(batch, M-s)
        Cp = 1 + rng.normal(0, sigma_r, (b, n_unit))
        Cn = 1 + rng.normal(0, sigma_r, (b, n_unit))
        T = np.sort((Cn @ Mn.T)/Cn.sum(1)[:, None] - (Cp @ Mp.T)/Cp.sum(1)[:, None], axis=1)
        lsb = (T[:, -1] - T[:, 0])[:, None]/(T.shape[1]-1)
        inl = (T - T[:, :1])/lsb - np.arange(T.shape[1])
        dnl = np.diff(T, axis=1)/lsb - 1.0
        acc.append(np.column_stack([dnl[:, i0-1], np.abs(dnl).max(1), dnl.min(1),
                                    np.sqrt((inl**2).mean(1)), np.abs(inl).max(1)]))
    a = np.vstack(acc)
    return dict(K_DNL_MID=float(a[:, 0].std()/sigma_r), K_DNL_MAX=float(a[:, 1].mean()/sigma_r),
                K_INL_MAX=float(a[:, 4].mean()/sigma_r), K_INL_RMS=float(a[:, 3].mean()/sigma_r),
                P_DNL_GT_HALF=float((a[:, 1] > 0.5).mean()), P_NONMONO=float((a[:, 2] < -1).mean()),
                P997_DNL=float(np.percentile(a[:, 1], 99.7)),
                P997_INL=float(np.percentile(a[:, 4], 99.7)), M=int(M))


Mp_, Mn_, grp_ = build_maps()
T_ideal = np.sort((np.ones(n_unit) @ Mn_.T - np.ones(n_unit) @ Mp_.T)/n_unit)
print(f'\ncek array ideal: {len(T_ideal)} threshold, '
      f'{T_ideal[0]/LSB:+.1f} .. {T_ideal[-1]/LSB:+.1f} LSB, '
      f'step min={np.diff(T_ideal).min()/LSB:.6f} max={np.diff(T_ideal).max()/LSB:.6f} LSB')

profil varians per level threshold -- kenapa rumus tangan itu salah:
  level m   n_threshold   n_unit terlibat = 2^(N-1)*(1-2^-(m-1))
     1             1             0
     2             2            64
     3             4            96
     4             8           112
     5            16           120
     6            32           124
     7            64           126
     8           128           127
  level 1 memakai NOL unit; hanya level 8 yang memakai 127.
  rata-rata berbobot = 123.98 unit

cek array ideal: 255 threshold, -127.0 .. +127.0 LSB, step min=1.000000 max=1.000000 LSB


In [6]:
K_ANALYTIC = np.sqrt(sum(w_step))
mcres = {sr: mc_linearity(sr, M=20000) for sr in (0.007, 0.014, 0.028)}

print(f'analitik  sigma_DNL,mid/sigma_r = sqrt({sum(w_step)}) = {K_ANALYTIC:.4f}\n')
print('sigma_r   K_DNL_MID   vs analitik   K_DNL_MAX   K_INL_MAX   K_INL_RMS')
for sr, r in mcres.items():
    print(f'  {sr*100:4.1f}%    {r["K_DNL_MID"]:7.4f}     {100*(r["K_DNL_MID"]/K_ANALYTIC-1):+6.1f}%     '
          f'{r["K_DNL_MAX"]:7.4f}     {r["K_INL_MAX"]:7.4f}     {r["K_INL_RMS"]:7.4f}')

K = mcres[0.014]
K_DNL_MID = K_ANALYTIC          # MC setuju -> pakai bentuk tertutup
K_DNL_MAX = K['K_DNL_MAX']
K_INL_MAX = K['K_INL_MAX']
K_INL_RMS = K['K_INL_RMS']
K_HAND_WRONG = np.sqrt(sum(w_step)*2/3)

print(f'\nkoefisien yang dipakai (semua per sigma_r, dalam LSB):')
print(f'  K_DNL_MID = {K_DNL_MID:7.4f}   analitik sqrt({sum(w_step)}), MC setuju -> dipakai')
print(f'  K_DNL_MAX = {K_DNL_MAX:7.4f}   MC saja')
print(f'  K_INL_MAX = {K_INL_MAX:7.4f}   MC saja')
print(f'  K_INL_RMS = {K_INL_RMS:7.4f}   MC saja')
print(f'\n  turunan tangan lama sqrt({sum(w_step)}*2/3) = {K_HAND_WRONG:.4f} '
      f'-> {100*(K_HAND_WRONG/K_INL_RMS-1):+.1f}% PESIMIS. Dibuang.')
print('  koefisien MC hanya berlaku untuk N, N_SPLIT, dan skema INI. Ganti salah satu,')
print('  jalankan ulang sel ini. Jangan bawa angkanya ke resolusi lain.')

analitik  sigma_DNL,mid/sigma_r = sqrt(127) = 11.2694

sigma_r   K_DNL_MID   vs analitik   K_DNL_MAX   K_INL_MAX   K_INL_RMS
   0.7%    11.2183       -0.5%     14.3307     13.5997      6.0044
   1.4%    11.2182       -0.5%     14.3307     13.5998      6.0044
   2.8%    11.2057       -0.6%     14.3205     13.5989      6.0044

koefisien yang dipakai (semua per sigma_r, dalam LSB):
  K_DNL_MID = 11.2694   analitik sqrt(127), MC setuju -> dipakai
  K_DNL_MAX = 14.3307   MC saja
  K_INL_MAX = 13.5998   MC saja
  K_INL_RMS =  6.0044   MC saja

  turunan tangan lama sqrt(127*2/3) = 9.2014 -> +53.2% PESIMIS. Dibuang.
  koefisien MC hanya berlaku untuk N, N_SPLIT, dan skema INI. Ganti salah satu,
  jalankan ulang sel ini. Jangan bawa angkanya ke resolusi lain.


## Langkah 5 — Pilih `C_u`

Dua kriteria. Perhatikan istilahnya: yang mengikat adalah **spec DNL 0.5 LSB**, bukan
batas monotonisitas. Monotonisitas butuh `DNL > -1 LSB`, jauh lebih longgar — MC di
Langkah 4 memberi `P(non-monotonic) = 0`.

```
(A) SPEC DNL      : 3*K_DNL_MID*sigma_r <= 0.5 LSB
(B) ANGGARAN ENOB : K_INL_RMS*sigma_r*LSB <= (jatah mismatch)
```

Karena `C_u` tidak boleh bergantung pada pembagian anggaran, kriteria (B) diuji pada
batas paling longgar: seluruh `V_circ`. Kalau ia tetap tidak mengikat, maka `C_u`
sepenuhnya ditentukan spec DNL — dan itu yang membuat notebook ini bisa berdiri sendiri.

In [7]:
sr_dnl = 0.5/(3*K_DNL_MID)
sr_enob = (Vcirc/LSB)/K_INL_RMS
print(f'(A) spec DNL   : sigma_r <= 0.5/(3*{K_DNL_MID:.4f}) = {sr_dnl*100:.6f} %'
      f'  -> A_u >= {(A_C/sr_dnl)**2:.6f} um^2  -> sisi >= {A_C/sr_dnl:.6f} um   <== MENGIKAT')
print(f'(B) ENOB       : sigma_r <= ({Vcirc/LSB:.6f} LSB)/{K_INL_RMS:.4f} = {sr_enob*100:.6f} %'
      f'  -> sisi >= {A_C/sr_enob:.6f} um   (di bawah DRC {CAPM_W_MIN} um)')
print(f'\n  -> C_u ditentukan 100% oleh spec DNL. Anggaran noise tidak ikut menentukan.\n')

print('sweep kandidat:')
print(' sisi(um)  A_u(um^2)  sigma_r(%)   C_u(fF)  C_tot(fF)  3*sDNL   E|DNL|  INL_rms  lolos')
cands = [1.0, 1.4, 1.8, 1.9, 2.0, 2.2, 2.7]
for s in cands:
    sr = sigma_unit(s); cu = c_unit(s)
    ok = 'YA' if 3*K_DNL_MID*sr <= 0.5 and s >= CAPM_W_MIN else 'tidak'
    print(f'  {s:5.2f}    {s*s:7.4f}   {sr*100:8.4f}  {cu*1e15:8.4f}  {n_unit*cu*1e15:8.2f}  '
          f'{3*K_DNL_MID*sr:7.4f}  {K_DNL_MAX*sr:7.4f}  {K_INL_RMS*sr:7.4f}   {ok}')

# ===== (c) PILIHAN =========================================================
CAPM_SIDE = 2.0
# 1.9 um lolos dengan margin 0.4% -- itu kebetulan, bukan margin. 2.0 memberi 5.3%
# dan angkanya bulat untuk layout, dengan biaya 143 um^2 luas tambahan.

C_u     = c_unit(CAPM_SIDE)
sigma_r = sigma_unit(CAPM_SIDE)
C_tot   = n_unit*C_u
mck     = mc_linearity(sigma_r, M=20000, seed=1)

print(f'\nPILIHAN: capm {CAPM_SIDE} x {CAPM_SIDE} um')
print(f'  C_u        = {CAMIMC*1e15:.2f}*{CAPM_SIDE}*{CAPM_SIDE} + {CPMIMC*1e15:.2f}*2*'
      f'({CAPM_SIDE}+{CAPM_SIDE}) = {CAMIMC*CAPM_SIDE**2*1e15:.4f} + '
      f'{CPMIMC*2*2*CAPM_SIDE*1e15:.4f} = {C_u*1e15:.4f} fF')
print(f'  suku perimeter = {100*CPMIMC*2*2*CAPM_SIDE/C_u:.1f} % dari C_u  '
      f'<- BUKAN koreksi kecil. Lihat Langkah 9.')
print(f'  sigma_r    = {A_C*100:.1f}/sqrt({CAPM_SIDE**2:.1f}) = {sigma_r*100:.6f} %')
print(f'  C_tot,sisi = {n_unit}*{C_u*1e15:.4f} = {C_tot*1e15:.2f} fF')
print(f'  C_array    = 2*{C_tot*1e15:.2f} = {2*C_tot*1e12:.5f} pF\n')
print(f'  3*sigma_DNL,mid = {3*K_DNL_MID*sigma_r:.6f} LSB <= 0.5   '
      f'margin {100*(0.5/(3*K_DNL_MID*sigma_r)-1):.1f} %')
print(f'  E[max|DNL|]     = {K_DNL_MAX*sigma_r:.6f} LSB      '
      f'E[max|INL|] = {K_INL_MAX*sigma_r:.6f} LSB')
print(f'  INL_rms         = {K_INL_RMS*sigma_r:.6f} LSB = {K_INL_RMS*sigma_r*LSB*1e3:.6f} mV')
print(f'  MC yield: P(max|DNL|>0.5 LSB) = {mck["P_DNL_GT_HALF"]*100:.2f} %   '
      f'P(non-monotonic) = {mck["P_NONMONO"]*100:.3f} %')
print(f'            p99.7 max|DNL| = {mck["P997_DNL"]:.4f} LSB   '
      f'p99.7 max|INL| = {mck["P997_INL"]:.4f} LSB')
print('\n  Kriteria 3-sigma di mid-code ternyata terkalibrasi hampir persis terhadap')
print('  yield max-over-codes yang sebenarnya. Itu kebetulan menguntungkan, bukan')
print('  turunan -- jadi selalu cek dengan MC kalau N atau N_SPLIT berubah.')

(A) spec DNL   : sigma_r <= 0.5/(3*11.2694) = 1.478928 %  -> A_u >= 3.584448 um^2  -> sisi >= 1.893264 um   <== MENGIKAT
(B) ENOB       : sigma_r <= (0.289035 LSB)/6.0044 = 4.813739 %  -> sisi >= 0.581668 um   (di bawah DRC 1.0 um)

  -> C_u ditentukan 100% oleh spec DNL. Anggaran noise tidak ikut menentukan.

sweep kandidat:
 sisi(um)  A_u(um^2)  sigma_r(%)   C_u(fF)  C_tot(fF)  3*sDNL   E|DNL|  INL_rms  lolos
   1.00     1.0000     2.8718    2.6422    338.21   0.9709   0.4115   0.1724   tidak
   1.40     1.9600     2.0364    4.8263    617.76   0.6885   0.2918   0.1223   tidak
   1.80     3.2400     1.5775    7.6503    979.23   0.5333   0.2261   0.0947   tidak
   1.90     3.6100     1.4933    8.4563   1082.40   0.5049   0.2140   0.0897   tidak
   2.00     4.0000     1.4177    9.3023   1190.69   0.4793   0.2032   0.0851   YA
   2.20     4.8400     1.2874   11.1143   1422.62   0.4352   0.1845   0.0773   YA
   2.70     7.2900     1.0467   16.3443   2092.06   0.3539   0.1500   0.0628   YA


PILIHAN: capm 2.0 x 2.0 um
  C_u        = 2.00*2.0*2.0 + 0.19*2*(2.0+2.0) = 8.0000 + 1.5200 = 9.3023 fF
  suku perimeter = 16.3 % dari C_u  <- BUKAN koreksi kecil. Lihat Langkah 9.
  sigma_r    = 2.8/sqrt(4.0) = 1.417722 %
  C_tot,sisi = 128*9.3023 = 1190.69 fF
  C_array    = 2*1190.69 = 2.38138 pF

  3*sigma_DNL,mid = 0.479307 LSB <= 0.5   margin 4.3 %
  E[max|DNL|]     = 0.203170 LSB      E[max|INL|] = 0.192807 LSB
  INL_rms         = 0.085125 LSB = 1.197075 mV
  MC yield: P(max|DNL|>0.5 LSB) = 0.38 %   P(non-monotonic) = 0.000 %
            p99.7 max|DNL| = 0.5069 LSB   p99.7 max|INL| = 0.4027 LSB

  Kriteria 3-sigma di mid-code ternyata terkalibrasi hampir persis terhadap
  yield max-over-codes yang sebenarnya. Itu kebetulan menguntungkan, bukan
  turunan -- jadi selalu cek dengan MC kalau N atau N_SPLIT berubah.


## Langkah 6 — Berapa anggaran error yang CDAC habiskan, dan apa yang tersisa

Tiga suku CDAC, semuanya **dihitung**, tidak ada yang dialokasikan:

```
mismatch    sigma_mm  = K_INL_RMS*sigma_r*LSB           <- DIPAKSA oleh spec DNL
kT/C        sigma_kTC = sqrt(2*k*T/C_tot)               <- gratis
driver      sigma_drv^2 = sum k*T*C_sw/(C_tot*C_rest)   <- gratis
```

Suku driver menarik: **`R_drv` hilang dari hasilnya**, karena ia menaikkan rapat noise dan
menurunkan bandwidth dengan faktor yang sama. Suku ini tidak bisa diperbaiki dengan
memperbesar switch, dan tidak perlu.

Sel terakhir menulis `cdac_result.json` — antarmuka ke `comparator.ipynb`.

In [8]:
sigma_mm  = K_INL_RMS*sigma_r*LSB
sigma_ktc = np.sqrt(2*kT/C_tot)

print('noise driver bottom plate, sigma^2 = k*T*C_sw/(C_tot*C_rest):')
var_drv = 0.0
for k, w in enumerate(w_step, start=1):
    c_sw = w//2 if k <= N_SPLIT else w
    n_dr = 2    if k <= N_SPLIT else 1
    v = n_dr*kT*c_sw/(n_unit*(n_unit-c_sw)*C_u)
    var_drv += v
    print(f'  k={k}: C_sw={c_sw:3d} C_rest={n_unit-c_sw:3d} C_u, {n_dr} driver -> '
          f'{np.sqrt(v/n_dr)*1e6:6.2f} uV tiap')
sigma_drv = np.sqrt(var_drv)
print(f'  total (dijumlah kuadrat) = {sigma_drv*1e6:.2f} uV = {sigma_drv/LSB:.5f} LSB')

sigma_cdac = np.sqrt(sigma_mm**2 + sigma_ktc**2 + sigma_drv**2)
C_par_budget = C_tot*(1/0.99 - 1)

print(f'\nkonsumsi anggaran CDAC (V_circ = {Vcirc*1e3:.4f} mV):')
for nm, v, note in [('mismatch (INL_rms)', sigma_mm,  'DIPAKSA spec DNL'),
                    ('kT/C sampling',      sigma_ktc, 'gratis'),
                    ('driver bottom plate', sigma_drv, 'gratis')]:
    print(f'  {nm:22s} {v*1e3:8.4f} mV  {100*v**2/Vcirc**2:5.1f} % daya   {note}')
print(f'  {"TOTAL CDAC":22s} {sigma_cdac*1e3:8.4f} mV  {100*sigma_cdac**2/Vcirc**2:5.1f} % daya')
print(f'  {"tersisa (kuadratur)":22s} '
      f'{np.sqrt(Vcirc**2-sigma_cdac**2)*1e3:8.4f} mV  <- untuk comparator + noise VREF + jitter')

res = dict(CAPM_SIDE=CAPM_SIDE, C_u=C_u, sigma_r=sigma_r, C_tot=C_tot, n_unit=n_unit,
           N_SPLIT=N_SPLIT, K_DNL_MID=K_DNL_MID, K_DNL_MAX=K_DNL_MAX,
           K_INL_MAX=K_INL_MAX, K_INL_RMS=K_INL_RMS,
           sigma_mm=sigma_mm, sigma_ktc=sigma_ktc, sigma_drv=sigma_drv,
           sigma_cdac=sigma_cdac, C_par_budget_1pct=C_par_budget,
           G_REF_last=0.5, G_REF_worst=0.9922,
           P_DNL_GT_HALF=mck['P_DNL_GT_HALF'], P_NONMONO=mck['P_NONMONO'],
           P997_DNL=mck['P997_DNL'], P997_INL=mck['P997_INL'], MC_M=mck['M'])
print('\nditulis ->', save_cdac(res))
print('  comparator.ipynb membaca ini lewat adc_spec.load_cdac(); tanpa file ini ia menolak jalan.')

noise driver bottom plate, sigma^2 = k*T*C_sw/(C_tot*C_rest):
  k=1: C_sw= 32 C_rest= 96 C_u, 2 driver ->  34.05 uV tiap
  k=2: C_sw= 16 C_rest=112 C_u, 2 driver ->  22.29 uV tiap
  k=3: C_sw=  8 C_rest=120 C_u, 2 driver ->  15.23 uV tiap
  k=4: C_sw=  4 C_rest=124 C_u, 2 driver ->  10.59 uV tiap
  k=5: C_sw=  4 C_rest=124 C_u, 1 driver ->  10.59 uV tiap
  k=6: C_sw=  2 C_rest=126 C_u, 1 driver ->   7.43 uV tiap
  k=7: C_sw=  1 C_rest=127 C_u, 1 driver ->   5.23 uV tiap
  total (dijumlah kuadrat) = 64.78 uV = 0.00461 LSB

konsumsi anggaran CDAC (V_circ = 4.0646 mV):
  mismatch (INL_rms)       1.1971 mV    8.7 % daya   DIPAKSA spec DNL
  kT/C sampling            0.0834 mV    0.0 % daya   gratis
  driver bottom plate      0.0648 mV    0.0 % daya   gratis
  TOTAL CDAC               1.2017 mV    8.7 % daya
  tersisa (kuadratur)      3.8828 mV  <- untuk comparator + noise VREF + jitter

ditulis -> /foss/designs/ttsky26c-saradc/sizing/cdac_result.json
  comparator.ipynb membaca ini lewat adc

## Langkah 7 — Parasitik top plate dan settling

Selama sampling top plate di-drive ke Vin, jadi parasitik `C_p` tidak menghadiri
pengambilan sinyal. Setelah switch membuka, ia ikut dalam pembagi muatan:

```
atenuasi = C_tot/(C_tot + C_p)
```

Semua langkah teratenuasi faktor **sama** -> **gain error, bukan INL**. Yang berbahaya
bukan besarnya `C_p` tapi **asimetrinya antara sisi-p dan sisi-n** — itu offset, dan tidak
muncul di persamaan mana pun di notebook ini. Ditangani layout (Langkah 9).

CDAC menerbitkan **anggaran** `C_p`; gerbang input comparator harus masuk ke dalamnya.
Arah itu penting supaya tidak ada ketergantungan melingkar antara dua notebook.

In [9]:
print(f'anggaran C_p yang diterbitkan CDAC (C_tot = {C_tot*1e15:.2f} fF):')
for err in (0.005, 0.01, 0.02, 0.05):
    print(f'  gain error < {err*100:4.1f} % : C_p <= C_tot*(1/{1-err:.3f}-1) = '
          f'{C_tot*(1/(1-err)-1)*1e15:7.2f} fF   (C_tot >= {1/(1/(1-err)-1):.0f}*C_p)')
print('  -> comparator.ipynb wajib menunjukkan CGG gerbang input pair masuk ke sini.')

n_tau_samp = np.log(FS_diff/(LSB/2))
print(f'\nsettling sampling switch:')
print(f'  n_tau = ln(FS_diff/(LSB/2)) = ln({FS_diff:.3f}/{LSB/2:.7f}) = ln({FS_diff/(LSB/2):.1f}) '
      f'= {n_tau_samp:.4f}')
for m in (6, 8, 10):
    print(f'  ambil {m:2d} tau -> R_on <= T_SAMPLE/({m}*C_tot) = '
          f'{T_SAMPLE/(m*C_tot):7.1f} ohm')

print(f'\nbeban driver bottom plate, C_seri = C_sw*(C_tot-C_sw)/C_tot:')
print('  k   C_sw(C_u)  C_seri(fF)   n_tau = ln(step/(LSB/2))')
for k, w in enumerate(w_step, start=1):
    c_sw = w//2 if k <= N_SPLIT else w
    ser = c_sw*(n_unit-c_sw)/n_unit*C_u
    print(f'  {k}     {c_sw:5d}     {ser*1e15:8.2f}     {np.log((VREF/2**k)/(LSB/2)):.4f}')
print('  parasitik bottom plate di-drive impedansi rendah -> hanya membebani driver,')
print('  tidak merusak rasio transfer.')

anggaran C_p yang diterbitkan CDAC (C_tot = 1190.69 fF):
  gain error <  0.5 % : C_p <= C_tot*(1/0.995-1) =    5.98 fF   (C_tot >= 199*C_p)
  gain error <  1.0 % : C_p <= C_tot*(1/0.990-1) =   12.03 fF   (C_tot >= 99*C_p)
  gain error <  2.0 % : C_p <= C_tot*(1/0.980-1) =   24.30 fF   (C_tot >= 49*C_p)
  gain error <  5.0 % : C_p <= C_tot*(1/0.950-1) =   62.67 fF   (C_tot >= 19*C_p)
  -> comparator.ipynb wajib menunjukkan CGG gerbang input pair masuk ke sini.

settling sampling switch:
  n_tau = ln(FS_diff/(LSB/2)) = ln(3.600/0.0070313) = ln(512.0) = 6.2383
  ambil  6 tau -> R_on <= T_SAMPLE/(6*C_tot) =  2099.6 ohm
  ambil  8 tau -> R_on <= T_SAMPLE/(8*C_tot) =  1574.7 ohm
  ambil 10 tau -> R_on <= T_SAMPLE/(10*C_tot) =  1259.8 ohm

beban driver bottom plate, C_seri = C_sw*(C_tot-C_sw)/C_tot:
  k   C_sw(C_u)  C_seri(fF)   n_tau = ln(step/(LSB/2))
  1        32       223.25     4.8520
  2        16       130.23     4.1589
  3         8        69.77     3.4657
  4         4        36.05 

## Langkah 8 — Energi referensi, dan kopling noise referensi

Muatan yang ditarik dari VREF per transisi, dengan `S` = himpunan cap yang bottom
plate-nya tersambung VREF:

```
dQ = sum(C_i tetap di VREF)*(V_top,sblm - V_top,ssdh)
   + sum(C_i baru ke VREF) *(VREF - V_top,ssdh + V_top,sblm)
E  = VREF*dQ
```

Sekaligus di sini dihitung **gain kopling noise VREF ke differensial**,
`G_REF = |C_S,p - C_S,n|/C_tot`. Suku ini dulu tidak punya baris di anggaran mana pun, dan
ia besar.

In [10]:
def convert(vip, vin, ns=N_SPLIT, track=False):
    grp, o = {}, 0
    for k in range(1, n_step+1):
        w = w_step[k-1]
        if k <= ns:
            grp[('A', k)] = (w/2, VREF); grp[('B', k)] = (w/2, 0.0)
        else:
            grp[('M', k)] = (w, VREF)
    grp[('D', 0)] = (w_dummy, 0.0)
    P = {kk: [v[0], v[1], v[1]] for kk, v in grp.items()}   # [bobot, B_now, B0]
    Nn = {kk: [v[0], v[1], v[1]] for kk, v in grp.items()}
    E = 0.0; g = []

    def vtop(S, v0): return v0 + sum(w*(b-b0)/n_unit for w, b, b0 in S.values())
    def cs(S):       return sum(w for w, b, _ in S.values() if b == VREF)

    def move(S, v0, keys, val):
        vb = vtop(S, v0); Cs = cs(S)
        Cnew = sum(S[k][0] for k in keys if val == VREF and S[k][1] != VREF)
        for k in keys: S[k][1] = val
        va = vtop(S, v0); Cstay = cs(S) - Cnew
        return VREF*(Cstay*(vb-va) + Cnew*(VREF-va+vb))*C_u

    for k in range(1, N+1):
        vp, vn = vtop(P, vip), vtop(Nn, vin)
        g.append(abs(cs(P)-cs(Nn))/n_unit)
        if k == N: break
        b = 1 if vp > vn else 0
        hi, lo = (P, Nn) if b else (Nn, P); vh, vl = (vip, vin) if b else (vin, vip)
        if k <= ns:
            E += move(hi, vh, [('A', k)], 0.0)
            E += move(lo, vl, [('B', k)], VREF)
        else:
            E += move(hi, vh, [('M', k)], 0.0)
    Er = sum(VREF*S[kk][0]*C_u*(VREF-v0)
             for S, v0 in ((P, vip), (Nn, vin)) for kk in S
             if S[kk][1] != VREF and S[kk][2] == VREF)
    return (E, Er, g) if track else (E, Er)


codes = np.arange(2**N)
vd = -VREF + (codes+0.5)*FS_diff/2**N
print('N_SPLIT   dV_CM(mV)   E_konv(fJ)  E_reset(fJ)  E_total(fJ)   P(uW)   vs mono')
base = None
for ns in (0, 3, 4, 5, 6):
    r = [convert(0.9+v/2, 0.9-v/2, ns) for v in vd]
    ec, er = np.mean([x[0] for x in r]), np.mean([x[1] for x in r])
    tot = ec + er
    if base is None: base = tot
    d = sum(VREF/2**(k+1) for k in range(ns+1, n_step))
    star = '  <==' if ns == N_SPLIT else ''
    print(f'   {ns}      {d*1e3:8.4f}   {ec*1e15:9.1f}   {er*1e15:9.1f}   {tot*1e15:9.1f}  '
          f'{tot*fs*1e6:7.2f}  {100*(tot/base-1):+6.1f} %{star}')
print('  Split LEBIH MURAH dari monotonic murni: preset split menaruh 67/128 unit di VREF')
print('  bukan 127/128, dan ayunan top plate per sisi cuma setengah.')

gm = [max(convert(0.9+v/2, 0.9-v/2, N_SPLIT, track=True)[2]) for v in vd]
gl = [convert(0.9+v/2, 0.9-v/2, N_SPLIT, track=True)[2][-1] for v in vd]
print(f'\ngain kopling VREF -> differensial, G_REF = |C_S,p - C_S,n|/C_tot:')
print(f'  saat sampling (preset simetris kedua sisi) : 0.0000  <- penolakan SEMPURNA')
print(f'  maksimum sepanjang konversi, rata-rata     : {np.mean(gm):.4f}  (worst {np.max(gm):.4f})')
print(f'  pada keputusan terakhir b{N}, rata-rata      : {np.mean(gl):.4f}  (worst {np.max(gl):.4f})')
print(f'\n  syarat: ripple_VREF,rms <= error_izin/G_REF')
for e in (0.1, 0.05):
    print(f'    error < {e:.2f} LSB -> ripple VREF rms <= {e*LSB/np.mean(gl)*1e3:.4f} mV '
          f'({20*np.log10(e*LSB/np.mean(gl)/VREF):.1f} dB relatif {VREF} V)')
print(f'\n  aperture jitter: sigma_jitter <= error_izin/(2*pi*F_IN*VREF), '
      f'slew = {2*np.pi*F_IN*VREF:.4e} V/s')
for e in (0.1, 0.05):
    print(f'    error < {e:.2f} LSB -> sigma_jitter <= {e*LSB/(2*np.pi*F_IN*VREF)*1e12:.2f} ps')

N_SPLIT   dV_CM(mV)   E_konv(fJ)  E_reset(fJ)  E_total(fJ)   P(uW)   vs mono
   0      885.9375      1883.9      1270.9      3154.9    31.55    +0.0 %


   3       98.4375      1277.4       743.5      2020.9    20.21   -35.9 %


   4       42.1875      1271.7       687.0      1958.7    19.59   -37.9 %  <==
   5       14.0625      1270.8       657.8      1928.6    19.29   -38.9 %


   6        0.0000      1270.8       642.9      1913.7    19.14   -39.3 %
  Split LEBIH MURAH dari monotonic murni: preset split menaruh 67/128 unit di VREF
  bukan 127/128, dan ayunan top plate per sisi cuma setengah.



gain kopling VREF -> differensial, G_REF = |C_S,p - C_S,n|/C_tot:
  saat sampling (preset simetris kedua sisi) : 0.0000  <- penolakan SEMPURNA
  maksimum sepanjang konversi, rata-rata     : 0.6666  (worst 0.9922)
  pada keputusan terakhir b8, rata-rata      : 0.5000  (worst 0.9922)

  syarat: ripple_VREF,rms <= error_izin/G_REF
    error < 0.10 LSB -> ripple VREF rms <= 2.8125 mV (-56.1 dB relatif 1.8 V)
    error < 0.05 LSB -> ripple VREF rms <= 1.4063 mV (-62.1 dB relatif 1.8 V)

  aperture jitter: sigma_jitter <= error_izin/(2*pi*F_IN*VREF), slew = 5.6549e+07 V/s
    error < 0.10 LSB -> sigma_jitter <= 24.87 ps
    error < 0.05 LSB -> sigma_jitter <= 12.43 ps


## Langkah 9 — Penyusunan: aritmetika yang menentukan layout

Model mismatch PDK hanya menangkap komponen **acak**. Semuanya di sini adalah error
**sistematis** yang tidak muncul di persamaan mana pun di atas.

1. **Unit cell wajib** — karena `C = camimc*A + cpmimc*2*(W+L)`, dua kapasitor berarea
   sama tapi bentuk beda punya nilai beda. Sel di bawah menghitung selisihnya.
2. **Pitch ditentukan capm.2b (1.20 um, spacing met3 bottom plate)**, bukan capm.2a
   (0.84 um, spacing capm). Unit dalam satu paruh berbagi bottom plate -> boleh 0.84;
   unit beda grup atau beda sisi -> wajib 1.20.
3. **Sisi-p dan sisi-n wajib berbagi centroid -> interleaved.** Berdampingan membuat
   `C_tot,p != C_tot,n` sistematis (LSB kedua sisi beda -> INL). Dicermin menggandakan
   selisihnya. Hanya interleaved yang benar. Harganya kopling `C_pn` antar top plate,
   dan itu jinak: untuk sinyal differential ia meng-atenuasi semua langkah sama besar
   -> gain error, kategori sama dengan `C_p`.
4. **Dummy ring** diikat GND (jangan floating: capm floating bisa menahan muatan).
5. **Grup 1-unit tidak bisa di-centroid-kan** -> taruh grup terkecil paling dekat pusat.
6. **met3 = bottom plate (driven, jinak), capm -> via3 -> met4 = top plate (sensitif).**
   Konflik TT: power strap met4 di atas array kopling langsung ke top plate.

In [11]:
m_test = 32
A_blok = m_test*CAPM_SIDE**2
s_blok = np.sqrt(A_blok)
C_units = m_test*C_u
C_blok  = CAMIMC*A_blok + CPMIMC*2*(2*s_blok)
print(f'unit cell wajib, dan ini aritmetika bukan gaya:')
print(f'  suku perimeter C_u = {CPMIMC*2*2*CAPM_SIDE*1e15:.4f} fF dari {C_u*1e15:.4f} fF = '
      f'{100*CPMIMC*2*2*CAPM_SIDE/C_u:.1f} %')
print(f'  {m_test} unit {CAPM_SIDE}x{CAPM_SIDE} paralel : {m_test}*{C_u*1e15:.4f} '
      f'= {C_units*1e15:.2f} fF')
print(f'  satu blok berarea sama    : sisi {s_blok:.6f} um -> '
      f'{CAMIMC*A_blok*1e15:.2f} + {CPMIMC*2*2*s_blok*1e15:.2f} = {C_blok*1e15:.2f} fF')
print(f'  selisih {100*(C_blok/C_units-1):+.1f} %  <- array biner yang digambar sebagai')
print(f'  persegi panjang berskala sudah rusak sebelum mismatch ikut bicara.')

pitch_in  = CAPM_SIDE + CAPM_SP
pitch_out = CAPM_SIDE + CAPM_SP_M3
print(f'\npitch:')
print(f'  dalam satu grup (bottom plate sama) : {CAPM_SIDE} + {CAPM_SP} = {pitch_in:.2f} um  (capm.2a)')
print(f'  antar grup / antar sisi             : {CAPM_SIDE} + {CAPM_SP_M3} = {pitch_out:.2f} um  (capm.2b) <- penentu')
print(f'  karena p/n interleaved, hampir semua batas adalah batas grup -> pakai {pitch_out:.2f} um')

n_act = 2*n_unit
side_act = int(np.ceil(np.sqrt(n_act)))
print(f'\ngrid ({n_act} unit aktif = {n_unit} sisi-p + {n_unit} sisi-n):')
for rings in (0, 1, 2):
    sd = side_act + 2*rings
    tot = sd*sd
    A = tot*pitch_out**2
    print(f'  {rings} ring dummy: {sd:2d}x{sd:2d} = {tot:3d} posisi ({tot-n_act:3d} dummy), '
          f'{np.sqrt(A):.1f} x {np.sqrt(A):.1f} um = {A:7.1f} um^2 = {100*A/TILE_UM2:5.2f} % tile')
print(f'  tile 1x2 = {TILE_UM2} um^2. Ambil 2 ring: luas murah di sini.')

print(f'\nurutan penempatan dari pusat ke luar (grup kecil di pusat, gradien terkecil):')
order = [('dummy', w_dummy)] + [(f'M{k}', w_step[k-1]) for k in range(n_step, N_SPLIT, -1)] \
        + [(f'A{k}/B{k}', w_step[k-1]) for k in range(N_SPLIT, 0, -1)]
for nm, w in order:
    print(f'  {nm:8s} {w:3d} C_u' + ('  (paruh ' + str(w//2) + ' + ' + str(w//2) + ')'
                                     if 'A' in nm else ''))

unit cell wajib, dan ini aritmetika bukan gaya:
  suku perimeter C_u = 1.5200 fF dari 9.3023 fF = 16.3 %
  32 unit 2.0x2.0 paralel : 32*9.3023 = 297.67 fF
  satu blok berarea sama    : sisi 11.313708 um -> 256.00 + 8.60 = 264.60 fF
  selisih -11.1 %  <- array biner yang digambar sebagai
  persegi panjang berskala sudah rusak sebelum mismatch ikut bicara.

pitch:
  dalam satu grup (bottom plate sama) : 2.0 + 0.84 = 2.84 um  (capm.2a)
  antar grup / antar sisi             : 2.0 + 1.2 = 3.20 um  (capm.2b) <- penentu
  karena p/n interleaved, hampir semua batas adalah batas grup -> pakai 3.20 um

grid (256 unit aktif = 128 sisi-p + 128 sisi-n):
  0 ring dummy: 16x16 = 256 posisi (  0 dummy), 51.2 x 51.2 um =  2621.4 um^2 =  7.27 % tile
  1 ring dummy: 18x18 = 324 posisi ( 68 dummy), 57.6 x 57.6 um =  3317.8 um^2 =  9.20 % tile
  2 ring dummy: 20x20 = 400 posisi (144 dummy), 64.0 x 64.0 um =  4096.0 um^2 = 11.36 % tile
  tile 1x2 = 36072 um^2. Ambil 2 ring: luas murah di sini.

urutan penem

## Daftar yang masih terbuka

| Slot | Cara mengisinya |
|---|---|
| `DVOS_DVCM` | Sweep DC `V_CM` 0.79..0.90 V pada `comparator.sch`, cari trip point tiap titik, ambil slope. Efek **sistematis**, tidak perlu Monte Carlo. Menentukan apakah `N_SPLIT=4` cukup |
| `RIPPLE_VREF` | Ukur/anggarkan ripple supply TT. Kopling `G_REF` dari Langkah 8 |
| `SIGMA_JITTER` | Jitter clock TT pada 10 MHz |
| `C_PAR_TOP` | PEX setelah layout. Anggarannya diterbitkan di Langkah 7 |
| gradien proses | Tidak ada di model PDK. Ditangani Langkah 9, bukan sizing |
| skew driver split | Dua bottom plate bergerak serentak di sisi berlawanan. Spec settling atau spec akurasi? Bergantung kapan comparator melatch |
| verifikasi MC ngspice | 256 instance `cap_mim_m3_1` dengan `MC_MM_SWITCH=1`. Sekarang model mismatch-nya diketahui, jadi ini bisa dijalankan sungguhan |
| `cdac.sch` | Cek ulang bobot MF, simetri paruh split, dan daftar port terhadap Langkah 1 dan 6 |